# Transformer Scaling — SAINT + FT-CUR (corrigido)

Re-run **apenas** dos modelos de atenção inter-instâncias. Correções aplicadas:
- **Ponto 2:** treino e predição agora têm memória medida **separadamente**
  (`fitVRAM` vs `predVRAM`), e o OOM de predição é capturado sem descartar as
  métricas de treino.
- **SAINT:** predição mantém o regime **batch-level** do treino (chunks de
  `batch_size`), conforme o design do paper (Somepalli et al., 2021) — a atenção
  inter-instâncias é sempre sobre as linhas do batch, nunca sobre o dataset todo.

**Variantes:** `SAINT_minibatch`, `SAINT_fullbatch`, `FTCUR_minibatch`, `FTCUR_mfixed_full`  
**N:** [500, 1000, 2000, 5000, 10 000, 20 000, 50 000]  
**Repetições:** 3 (mediana)

**Saída:** `results/transformer_scaling_saint_ftcur.json` (arquivo separado — **não**
sobrescreve `transformer_scaling.json`).

**O que esperar (memória de treino e predição medidas à parte):**
- `SAINT_fullbatch` — atenção inter-instâncias densa e **global** → VRAM de
  **treino** O(N²) → OOM em N alto (variante de estresse, não o SAINT canônico).
- `SAINT_minibatch` — atenção inter-instâncias **batch-level** (Somepalli et al.,
  2021): cada linha atende só ao seu batch, no treino E na predição → O(B²)
  **constante**, escala em N. Contexto é **local ao batch**.
- `FTCUR_minibatch` — treino batch-local O(B·m); **predição global** O(N·m)
  (Nyström aproxima a atenção global com m landmarks) → escala sem OOM.
- `FTCUR_mfixed_full` — treino e predição globais O(N·m), m fixo → escala sem OOM.

**Antes de rodar:** Settings → Accelerator → GPU T4 x2 (ou P100).

In [ ]:
# ── Célula 1: Verifica GPU ───────────────────────────────────────────
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memória: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('GPU não detectada — ative em Settings > Accelerator')

In [ ]:
# ── Célula 2: Clonar repo ──────────────────────────────────────────
import os, subprocess

GIT_URL     = 'https://github.com/PauloBernardo/dissertacao-estudo-comparativo.git'
PROJECT_DIR = '/kaggle/working/sparse-lssvm-transformers-study'

if os.path.exists(PROJECT_DIR):
    subprocess.run(['git', '-C', PROJECT_DIR, 'pull', '--rebase'], check=True)
else:
    subprocess.run(['git', 'clone', GIT_URL, PROJECT_DIR], check=True)

os.chdir(PROJECT_DIR)
!git log --oneline -3
print(f'Dir: {os.getcwd()}')

In [ ]:
# ── Célula 3: Dependências ─────────────────────────────────────────
!pip install -q entmax einops psutil
import torch, sklearn, numpy, psutil
print(f'torch {torch.__version__} | sklearn {sklearn.__version__} | numpy {numpy.__version__} | psutil {psutil.__version__}')

In [ ]:
# ── Célula 4: Rodar benchmark (só SAINT + FT-CUR corrigidos) ────────────────
# Saída em arquivo SEPARADO para não sobrescrever o run completo.
# --variants restringe às 4 variantes inter-instâncias.
!python -u scripts/run_transformer_scaling.py \
    --output results/transformer_scaling_saint_ftcur.json \
    --repeats 3 \
    --variants SAINT_minibatch,SAINT_fullbatch,FTCUR_minibatch,FTCUR_mfixed_full \
    2>&1 | tee /kaggle/working/scaling_saint_ftcur.log

import shutil
shutil.copy('results/transformer_scaling_saint_ftcur.json',
            '/kaggle/working/transformer_scaling_saint_ftcur.json')
print('\nSalvo em /kaggle/working/transformer_scaling_saint_ftcur.json')

In [ ]:
# ── Célula 5: Resumo — treino vs predição (memória separada) ───────────────
import json
from pathlib import Path

records = json.loads(Path('results/transformer_scaling_saint_ftcur.json').read_text())

print(f"{'Variant':<20} {'N':>6} {'fit_s':>7} {'fitVRAM':>8} {'pred_ms':>9} {'predVRAM':>9}")
print('-' * 66)
for r in records:
    if r.get('skipped'):
        mk = 'OOM' if r.get('oom') else '—'
        print(f"{r['variant']:<20} {r['n']:>6} {mk:>7} {mk:>8} {mk:>9} {mk:>9}")
    else:
        pms  = r.get('pred_ms_median')
        pvr  = r.get('pred_vram_mb_median')
        pms_s  = 'OOM' if pms is None else f"{pms:.1f}ms"
        pvr_s  = 'OOM' if pvr is None else f"{pvr:.0f}MB"
        print(f"{r['variant']:<20} {r['n']:>6} {r['fit_s_median']:>6.1f}s "
              f"{r['vram_mb_median']:>6.0f}MB {pms_s:>9} {pvr_s:>9}")

print('\nLegenda: fitVRAM = pico VRAM no TREINO | predVRAM = pico VRAM na PREDIÇÃO')
print('- SAINT_fullbatch: fitVRAM O(N²) → OOM no TREINO (atenção densa global).')
print('- SAINT_minibatch: fitVRAM e predVRAM ~constantes O(B²) → contexto local ao batch.')
print('- FTCUR_*: predVRAM cresce ~linear O(N·m) → contexto global a custo linear.')